# Final 4-stage BSD35k filtered datasets

This notebook creates four BSD35k candidate datasets for team handoff.

- A: `v4_ge4` as the safest validated dataset.
- B: `v4_ge3` as a wider coverage candidate.
- C: `v4_ge4 + sp-p` expansion.
- D: `v4_ge4 + sp-p + sp-c` probe expansion.

Expansion rows are selected from BSD35k rows not already in `v4_ge4`, using v3 high-confidence probability and label-quality filtering. Class caps are based on BSD10k train_pool class counts.

In [ ]:
from pathlib import Path
import json
import math

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'baseline_confidnce_train').exists():
    ROOT = ROOT.parent

V4_DATA_DIR = ROOT / 'baseline_confidnce_train' / 'outputs' / 'v4_35k_hier_loss' / 'datasets'
SPLIT_PATH = ROOT / 'baseline_confidnce_train' / 'outputs' / 'v4_35k_hier_loss' / 'fixed_train_pool_final_test_split.csv'
V3_PATH = ROOT / 'outputs' / 'confidence_filter_v3' / 'predictions' / 'BSD35k-CS_filter_predictions.csv'
LABEL_QUALITY_PATH = ROOT / 'experiments' / 'bsd35k_label_quality' / 'BSD35k-CS_label_quality_scores.csv'

OUTPUT_DIR = ROOT / 'baseline_confidnce_train' / 'outputs' / 'final_4stage_filtered_datasets'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Conservative defaults. v3 0.40 is the F1-optimal threshold from the binary filter.
# Raising this to 0.70 is too strict for sp-p/sp-c expansion after removing v4_ge4 rows.
V3_PROB_THRESHOLD = 0.40
LABEL_QUALITY_THRESHOLD = 0.35
SP_P_CAP_RATIO = 1.00
SP_C_CAP_RATIO = 0.50  # use 0.25 for a stricter probe

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

## 1. Load source tables

In [ ]:
all_usable = pd.read_csv(V4_DATA_DIR / 'bsd35k_all_usable.csv')
v4_ge3 = pd.read_csv(V4_DATA_DIR / 'bsd35k_v4_ge3.csv')
v4_ge4 = pd.read_csv(V4_DATA_DIR / 'bsd35k_v4_ge4.csv')
v3 = pd.read_csv(V3_PATH)
label_quality = pd.read_csv(LABEL_QUALITY_PATH)
split_df = pd.read_csv(SPLIT_PATH)

for df in [all_usable, v4_ge3, v4_ge4, v3, label_quality, split_df]:
    if 'sound_id' in df.columns:
        df['sound_id'] = df['sound_id'].astype(str)
    if 'index' in df.columns:
        df['index'] = df['index'].astype(str)

train_pool = split_df[split_df['split'].eq('train_pool')].copy()
train_class_counts = train_pool['class'].value_counts().to_dict()

print('all usable:', len(all_usable))
print('v4_ge3:', len(v4_ge3))
print('v4_ge4:', len(v4_ge4))
print('train classes:', train_pool['class'].nunique())
print('sp-p train count:', train_class_counts.get('sp-p', 0))
print('sp-c train count:', train_class_counts.get('sp-c', 0))

## 2. Merge v3 and label-quality scores

In [ ]:
v3_cols = [
    'sound_id',
    'predicted_high_confidence_prob',
    'predicted_high_confidence',
    'predicted_high_confidence_strict',
    'binary_f1_optimal_threshold',
    'binary_precision_optimal_recall70_threshold',
]
lq_cols = [
    'sound_id',
    'label_quality_score',
    'label_noise_score',
    'quality_group',
    'same_class',
    'same_top_class',
    'provided_class_probability',
    'classifier_margin',
    'recommended_action',
]

score_df = all_usable.merge(v3[v3_cols], on='sound_id', how='left')
score_df = score_df.merge(label_quality[lq_cols], on='sound_id', how='left')

missing_v3 = score_df['predicted_high_confidence_prob'].isna().sum()
missing_lq = score_df['label_quality_score'].isna().sum()
print('missing v3 scores:', missing_v3)
print('missing label quality:', missing_lq)

score_df.head()

## 3. Select controlled expansion rows

In [ ]:
v4_ge4_ids = set(v4_ge4['sound_id'].astype(str))

def select_class_expansion(class_name: str, cap_ratio: float) -> pd.DataFrame:
    cap = math.floor(train_class_counts.get(class_name, 0) * cap_ratio)
    candidates = score_df[
        (~score_df['sound_id'].isin(v4_ge4_ids))
        & score_df['class'].eq(class_name)
        & score_df['predicted_high_confidence_prob'].ge(V3_PROB_THRESHOLD)
        & score_df['label_quality_score'].ge(LABEL_QUALITY_THRESHOLD)
    ].copy()
    candidates = candidates.sort_values(
        ['predicted_high_confidence_prob', 'label_quality_score', 'v4_filter_score'],
        ascending=False,
    )
    selected = candidates.head(cap).copy()
    selected['expansion_class_cap_ratio'] = cap_ratio
    selected['expansion_class_cap'] = cap
    selected['expansion_rank_within_class'] = range(1, len(selected) + 1)
    print(
        class_name,
        'cap_ratio=', cap_ratio,
        'cap=', cap,
        'candidates=', len(candidates),
        'selected=', len(selected),
    )
    return selected

sp_p_extra = select_class_expansion('sp-p', SP_P_CAP_RATIO)
sp_c_extra = select_class_expansion('sp-c', SP_C_CAP_RATIO)

display(sp_p_extra[['sound_id', 'class', 'predicted_high_confidence_prob', 'label_quality_score', 'v4_filter_score']].head())
display(sp_c_extra[['sound_id', 'class', 'predicted_high_confidence_prob', 'label_quality_score', 'v4_filter_score']].head())

## 4. Build the four datasets

In [ ]:
DCASE_COLS = [
    'sound_id', 'class', 'top_class', 'v4_filter_score', 'binary_mlp_prob',
    'fiveclass_score', 'fiveclass_p45', 'index', 'class_idx', 'top_class_idx',
    'audio_emb_filepath', 'text_emb_filepath', 'predicted_confidence_score',
]

def attach_scores(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    out = df.copy()
    out['sound_id'] = out['sound_id'].astype(str)
    if 'predicted_high_confidence_prob' not in out.columns:
        extra_cols = ['sound_id', 'predicted_high_confidence_prob', 'label_quality_score', 'label_noise_score', 'quality_group', 'recommended_action']
        out = out.merge(score_df[extra_cols], on='sound_id', how='left')
    out['dataset_source'] = source_name
    return out

def dedupe_by_sound_id(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates('sound_id', keep='first').reset_index(drop=True)

dataset_a = attach_scores(v4_ge4, 'A_v4_ge4')
dataset_b = attach_scores(v4_ge3, 'B_v4_ge3')
dataset_c = dedupe_by_sound_id(pd.concat([attach_scores(v4_ge4, 'base_v4_ge4'), sp_p_extra], ignore_index=True))
dataset_c['dataset_source'] = dataset_c['dataset_source'].fillna('C_sp_p_extra')
dataset_d = dedupe_by_sound_id(pd.concat([attach_scores(v4_ge4, 'base_v4_ge4'), sp_p_extra, sp_c_extra], ignore_index=True))
dataset_d['dataset_source'] = dataset_d['dataset_source'].fillna('D_sp_p_or_sp_c_extra')

datasets = {
    'A_v4_ge4': dataset_a,
    'B_v4_ge3': dataset_b,
    'C_v4_ge4_plus_sp_p': dataset_c,
    'D_v4_ge4_plus_sp_p_sp_c_probe': dataset_d,
}

for name, df in datasets.items():
    print(name, len(df), df['class'].nunique())
    display(df['class'].value_counts().rename('count').head(10).to_frame())

## 5. Save outputs

In [ ]:
manifest_rows = []

for name, df in datasets.items():
    full_path = OUTPUT_DIR / f'{name}_with_scores.csv'
    dcase_path = OUTPUT_DIR / f'{name}_dcase_rows.csv'
    counts_path = OUTPUT_DIR / f'{name}_class_counts.csv'

    df.to_csv(full_path, index=False)
    df[DCASE_COLS].to_csv(dcase_path, index=False)
    df['class'].value_counts().sort_index().rename('count').to_csv(counts_path)

    manifest_rows.append({
        'dataset_name': name,
        'rows': len(df),
        'classes': df['class'].nunique(),
        'sp_p_rows': int(df['class'].eq('sp-p').sum()),
        'sp_c_rows': int(df['class'].eq('sp-c').sum()),
        'full_with_scores_csv': str(full_path),
        'dcase_rows_csv': str(dcase_path),
        'class_counts_csv': str(counts_path),
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / 'dataset_manifest.csv', index=False)

config = {
    'v3_prob_threshold': V3_PROB_THRESHOLD,
    'label_quality_threshold': LABEL_QUALITY_THRESHOLD,
    'sp_p_cap_ratio': SP_P_CAP_RATIO,
    'sp_c_cap_ratio': SP_C_CAP_RATIO,
    'sp_p_train_count': int(train_class_counts.get('sp-p', 0)),
    'sp_c_train_count': int(train_class_counts.get('sp-c', 0)),
    'output_dir': str(OUTPUT_DIR),
}
(OUTPUT_DIR / 'selection_config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

display(manifest)
print('saved to:', OUTPUT_DIR)